In [9]:
"""
Builds NFHS-1 and NFHS-2 district -> 1991 Census district crosswalks using
fuzzy name matching within state. Falls back to the 1981 Census sheet for
states with no 1991 entry (e.g. Jammu & Kashmir).
 
INPUT FILES REQUIRED (in data/raw/DHS_Districts/):
    NFHS-1_DistrictCodes.xlsx     (cluster -> district lookup, NFHS-1)
    NFHS-2_DistrictCodes.xlsx     (cluster -> district lookup, NFHS-2)
    dist_list_81_91.xlsx          (1981 & 1991 Census district reference)
    District_codes-Phase1.xls     (per-state code->name, Phase 1 states)
    District_codes-Phase2.xls     (per-state code->name, Phase 2 states)
 
OUTPUT:
    data/processed/district_crosswalks/nfhs1_district_crosswalk.csv
    data/processed/district_crosswalks/nfhs2_district_crosswalk.csv
    data/processed/district_crosswalks/crosswalk_review_flags.csv
        (matches below MATCH_THRESHOLD)
"""

'\nBuilds NFHS-1 and NFHS-2 district -> 1991 Census district crosswalks using\nfuzzy name matching within state. Falls back to the 1981 Census sheet for\nstates with no 1991 entry (e.g. Jammu & Kashmir).\n\nINPUT FILES REQUIRED (in data/raw/DHS_Districts/):\n    NFHS-1_DistrictCodes.xlsx     (cluster -> district lookup, NFHS-1)\n    NFHS-2_DistrictCodes.xlsx     (cluster -> district lookup, NFHS-2)\n    dist_list_81_91.xlsx          (1981 & 1991 Census district reference)\n    District_codes-Phase1.xls     (per-state code->name, Phase 1 states)\n    District_codes-Phase2.xls     (per-state code->name, Phase 2 states)\n\nOUTPUT:\n    data/processed/district_crosswalks/nfhs1_district_crosswalk.csv\n    data/processed/district_crosswalks/nfhs2_district_crosswalk.csv\n    data/processed/district_crosswalks/crosswalk_review_flags.csv\n        (matches below MATCH_THRESHOLD)\n'

In [10]:
import pandas as pd
from rapidfuzz import process, fuzz
import re
from pathlib import Path
from manual_overrides import MANUAL_OVERRIDES

# Resolve repo root whether the notebook cwd is scripts/ or the project root
_cwd = Path.cwd()
if (_cwd / "data" / "raw" / "DHS_Districts").is_dir():
    REPO_ROOT = _cwd
elif (_cwd.parent / "data" / "raw" / "DHS_Districts").is_dir():
    REPO_ROOT = _cwd.parent
else:
    raise FileNotFoundError(
        "Could not find data/raw/DHS_Districts from cwd "
        f"{_cwd}. Run from the repo root or scripts/."
    )

RAW_DIR = REPO_ROOT / "data" / "raw" / "DHS_Districts"
OUT_DIR = REPO_ROOT / "data" / "processed" / "district_crosswalks"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MATCH_THRESHOLD = 90  # below this score, flag for manual review
print(f"RAW_DIR={RAW_DIR}")
print(f"OUT_DIR={OUT_DIR}")

RAW_DIR=/Users/eknoorsandhu/DHS_India_Research/data/raw/DHS_Districts
OUT_DIR=/Users/eknoorsandhu/DHS_India_Research/data/processed/district_crosswalks


In [11]:
# ---------------------------------------------------------------------------
# State name normalization: maps the messy variants in each source file to
# one canonical key so we can join state-by-state. Extend this if you hit
# a state that doesn't resolve (the script will print unmatched states).
# ---------------------------------------------------------------------------
STATE_ALIASES = {
    "andhra pradesh": "andhra pradesh",
    "arunachalpradesh": "arunachal pradesh",
    "arunachal pradesh": "arunachal pradesh",
    "assam": "assam",
    "bihar": "bihar",
    "goa": "goa",
    "gujarat": "gujarat",
    "haryana": "haryana",
    "himachal pradesh": "himachal pradesh",
    "himachal": "himachal pradesh",
    "jammu": "jammu & kashmir",
    "jammu & kashmir": "jammu & kashmir",
    "j & k": "jammu & kashmir",
    "karnataka": "karnataka",
    "kerala": "kerala",
    "madhya pradesh": "madhya pradesh",
    "maharashtra": "maharashtra",
    "manipur": "manipur",
    "meghalaya": "meghalaya",
    "mizoram": "mizoram",
    "nagaland": "nagaland",
    "new delhi": "delhi",
    "delhi": "delhi",
    "orissa": "orissa",
    "punjab": "punjab",
    "rajasthan": "rajasthan",
    "sikkim": "sikkim",
    "tamil nadu": "tamil nadu",
    "tripura": "tripura",
    "uttar pradesh": "uttar pradesh",
    "west bengal": "west bengal",
    # Census-only union territories / abbreviated 1981 labels -- these never
    # appear in the NFHS-1/2 state lists, kept here only to silence warnings
    # and allow the 1981-sheet fallback groupby to run cleanly.
    "andaman & nicobar": "andaman & nicobar",
    "andaman": "andaman & nicobar",
    "chandigarh": "chandigarh",
    "dadra + nagar haveli": "dadra & nagar haveli",
    "dadra +": "dadra & nagar haveli",
    "daman & diu": "daman & diu",
    "laksdadweep": "lakshadweep",
    "laccadive +": "lakshadweep",
    "pondicherry": "pondicherry",
    "andhra": "andhra pradesh",
    "arunachal": "arunachal pradesh",
    "goa +": "goa",
}

In [12]:
def norm_state(s):
    key = str(s).strip().lower()
    if key not in STATE_ALIASES:
        print(f"  [WARN] Unrecognized state name '{s}' -- add to STATE_ALIASES")
        return key
    return STATE_ALIASES[key]
 
 
def norm_district(s):
    """Lowercase, strip parentheticals and extra whitespace for fuzzy matching."""
    s = str(s).strip().lower()
    s = re.sub(r"\(.*?\)", "", s)      # drop "(ongole)" style parentheticals
    s = re.sub(r"[^a-z\s]", "", s)      # drop punctuation/ampersands
    s = re.sub(r"\s+", " ", s).strip()
    return s
 
 
def build_census_lookup(census91_df, census81_df):
    """
    Returns dict: canonical_state -> list of (raw_dlabel, normalized_dlabel, distid)
    Falls back to 1981 sheet for states absent from the 1991 sheet.
    """
    lookup = {}
    for df, tag in [(census91_df, "1991"), (census81_df, "1981")]:
        for state_raw, group in df.groupby("slabel"):
            state = norm_state(state_raw)
            if state in lookup:
                continue  # already have 1991 version, skip 1981 fallback
            lookup[state] = {
                "source_year": tag,
                "districts": [
                    (row["dlabel"], norm_district(row["dlabel"]), row["distid"])
                    for _, row in group.iterrows()
                ],
            }
    return lookup

In [13]:
def match_round(nfhs_df, census_lookup, round_label):
    """
    For each (state, shdist) in the NFHS crosswalk file, fuzzy-match the
    district name against the Census district list for that state.
    """
    results = []
    flags = []
 
    nfhs_districts = (
        nfhs_df[["hv024", "state_name", "shdist", "District"]]
        .drop_duplicates()
        .copy()
    )
 
    for _, row in nfhs_districts.iterrows():
        state = norm_state(row["state_name"])
        nfhs_dist_raw = row["District"]
        nfhs_dist_norm = norm_district(nfhs_dist_raw)
 
        census_entry = census_lookup.get(state)
        if census_entry is None:
            results.append({
                "round": round_label,
                "hv024": row["hv024"],
                "state": state,
                "shdist": row["shdist"],
                "nfhs_district_name": nfhs_dist_raw,
                "matched_census_district": None,
                "census_distid": None,
                "census_source_year": None,
                "match_score": 0,
            })
            flags.append({**results[-1], "reason": "no census entry for state"})
            continue
 
        choices = [d[1] for d in census_entry["districts"]]
        best = process.extractOne(nfhs_dist_norm, choices, scorer=fuzz.WRatio)
 
        if best is None:
            match_name, score, idx = None, 0, None
        else:
            match_name, score, idx = best
 
        if idx is not None:
            matched_raw, _, matched_distid = census_entry["districts"][idx]
        else:
            matched_raw, matched_distid = None, None
 
        record = {
            "round": round_label,
            "hv024": row["hv024"],
            "state": state,
            "shdist": row["shdist"],
            "nfhs_district_name": nfhs_dist_raw,
            "matched_census_district": matched_raw,
            "census_distid": matched_distid,
            "census_source_year": census_entry["source_year"],
            "match_score": score,
            "match_source": "fuzzy",
        }

        # Apply manual override if this (state, shdist, name) was hand-reviewed
        override_key = (state, row["shdist"], nfhs_dist_raw)
        if override_key in MANUAL_OVERRIDES:
            record["census_distid"] = MANUAL_OVERRIDES[override_key]
            record["match_score"] = 100
            record["match_source"] = "manual_override"

        # NE states with no real district breakdown in dist_list_81_91.xlsx --
        NE_STATES_NO_DISTRICT_DATA = {"manipur", "meghalaya", "mizoram", "nagaland"}
        if state in NE_STATES_NO_DISTRICT_DATA:
            record["census_distid"] = None
            record["match_source"] = "unresolved_no_source_data"

        results.append(record)

        if state in NE_STATES_NO_DISTRICT_DATA:
            flags.append({**record, "reason": "no real district breakdown in census source (NE state)"})
        elif record["match_score"] < MATCH_THRESHOLD:
            flags.append({**record, "reason": "low fuzzy match score"})
 
    return pd.DataFrame(results), pd.DataFrame(flags)
 
 
def main():
    print("Loading raw files...")
    nfhs1 = pd.ExcelFile(f"{RAW_DIR}/NFHS-1_DistrictCodes.xlsx").parse("District Codes")
    nfhs1 = nfhs1.rename(columns={"hv024.1": "state_name"})
 
    nfhs2 = pd.ExcelFile(f"{RAW_DIR}/NFHS-2_DistrictCodes.xlsx").parse("District Codes")
    nfhs2 = nfhs2.rename(columns={"hv024.1": "state_name"})
 
    census91 = pd.ExcelFile(f"{RAW_DIR}/dist_list_81_91.xlsx").parse("1991 Districts")
    census81 = pd.ExcelFile(f"{RAW_DIR}/dist_list_81_91.xlsx").parse("1981 Districts")
 
    print("Building Census lookup (1991 primary, 1981 fallback)...")
    census_lookup = build_census_lookup(census91, census81)
 
    print("Matching NFHS-1 districts...")
    xwalk1, flags1 = match_round(nfhs1, census_lookup, "NFHS-1")
 
    print("Matching NFHS-2 districts...")
    xwalk2, flags2 = match_round(nfhs2, census_lookup, "NFHS-2")

    # Global district key -- census_distid alone is only unique WITHIN a state
    # (same issue as the original shdist), so build a state-qualified key here
    # before export. This is the join key to use when building the panel.
    for xwalk in (xwalk1, xwalk2):
        xwalk["district_uid"] = xwalk["state"] + "_" + xwalk["census_distid"].astype("Int64").astype(str)
        xwalk.loc[xwalk["census_distid"].isna(), "district_uid"] = None

    xwalk1.to_csv(f"{OUT_DIR}/nfhs1_district_crosswalk.csv", index=False)
    xwalk2.to_csv(f"{OUT_DIR}/nfhs2_district_crosswalk.csv", index=False)
 
    all_flags = pd.concat([flags1, flags2], ignore_index=True)
    all_flags.to_csv(f"{OUT_DIR}/crosswalk_review_flags.csv", index=False)
 
    print("\n--- SUMMARY ---")
    print(f"NFHS-1: {len(xwalk1)} state-district rows, "
          f"{(xwalk1['match_score'] < MATCH_THRESHOLD).sum()} flagged for review "
          f"(avg score {xwalk1['match_score'].mean():.1f})")
    print(f"NFHS-2: {len(xwalk2)} state-district rows, "
          f"{(xwalk2['match_score'] < MATCH_THRESHOLD).sum()} flagged for review "
          f"(avg score {xwalk2['match_score'].mean():.1f})")
    print(f"\nOutputs written to {OUT_DIR}/")
    print("Review crosswalk_review_flags.csv before trusting the full panel merge.")
 
 
if __name__ == "__main__":
    main()

Loading raw files...
Building Census lookup (1991 primary, 1981 fallback)...
Matching NFHS-1 districts...
Matching NFHS-2 districts...

--- SUMMARY ---
NFHS-1: 393 state-district rows, 35 flagged for review (avg score 94.1)
NFHS-2: 440 state-district rows, 43 flagged for review (avg score 93.3)

Outputs written to /Users/eknoorsandhu/DHS_India_Research/data/processed/district_crosswalks/
Review crosswalk_review_flags.csv before trusting the full panel merge.


In [14]:
# Inspect raw NFHS state_name labels for Jammu & Kashmir
# (nfhs_districts is local to match_round(); rebuild it here from the source files)
nfhs1 = pd.ExcelFile(RAW_DIR / "NFHS-1_DistrictCodes.xlsx").parse("District Codes")
nfhs1 = nfhs1.rename(columns={"hv024.1": "state_name"})
nfhs2 = pd.ExcelFile(RAW_DIR / "NFHS-2_DistrictCodes.xlsx").parse("District Codes")
nfhs2 = nfhs2.rename(columns={"hv024.1": "state_name"})

for label, df in [("NFHS-1", nfhs1), ("NFHS-2", nfhs2)]:
    nfhs_districts = (
        df[["hv024", "state_name", "shdist", "District"]]
        .drop_duplicates()
        .copy()
    )
    jammu_names = nfhs_districts[
        nfhs_districts["state_name"].str.contains("jammu", case=False, na=False)
    ]["state_name"].unique()
    print(f"{label}: {jammu_names}")

NFHS-1: <ArrowStringArray>
['jammu']
Length: 1, dtype: str
NFHS-2: <ArrowStringArray>
['jammu']
Length: 1, dtype: str


In [15]:
x1 = pd.read_csv(OUT_DIR / "nfhs1_district_crosswalk.csv")
print(x1['district_uid'].isna().sum(), "rows with null district_uid")  # should equal NE row count
print(x1['district_uid'].duplicated().sum(), "duplicate district_uid values")  # should be 0

20 rows with null district_uid
36 duplicate district_uid values


In [16]:
import pandas as pd

NE_STATE_NAMES = ["arunachal pradesh", "manipur", "meghalaya", "mizoram", "nagaland"]

for round_label, path in [
    ("NFHS-1", "data/raw/DHS_microdata/NFHS1_1992-93_IndividualRecode/IAIR23FL.DTA"),
    ("NFHS-2", "data/raw/DHS_microdata/NFHS2_1998-99_IndividualRecode/IAIR42FL.DTA"),
]:
    df = pd.read_stata(path, convert_categoricals=True)
    # adjust column names below to match your actual variable names
    df["state_norm"] = df["hv024"].astype(str).str.strip().str.lower()
    subset = df[df["state_norm"].isin(NE_STATE_NAMES)]
    pairs = (
        subset[["state_norm", "shdist"]]
        .drop_duplicates()
        .sort_values(["state_norm", "shdist"])
    )
    print(f"--- {round_label} ---")
    print(pairs.to_string(index=False))

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/DHS_microdata/NFHS1_1992-93_IndividualRecode/IAIR23FL.DTA'